# Rwanda Water Meter — YOLOv8s Training for ≥95% Accuracy

## Before you start
1. **Enable GPU**: Runtime → Change runtime type → T4 GPU → Save
2. **Upload your dataset folder to Google Drive** (see Step 2 below — no zipping needed)

---

## Step 1 — Check GPU and install dependencies

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU')

!pip install ultralytics -q
import ultralytics
print(f'Ultralytics version: {ultralytics.__version__}')

## Step 2 — Upload your dataset folder to Google Drive

**Do this once on your PC before running this cell:**
1. Open [drive.google.com](https://drive.google.com) in your browser
2. Click **New → Folder upload**
3. Navigate to your project folder and select the **`dataset`** folder
4. Wait for it to finish uploading (it will appear as `dataset` in My Drive)

That's it — no zipping needed. Then run the cell below.

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

# Path to the dataset folder you uploaded in Google Drive
# If you uploaded it somewhere other than the root of My Drive, update this path
GDRIVE_DATASET = '/content/drive/MyDrive/dataset'

if not os.path.exists(GDRIVE_DATASET):
    raise FileNotFoundError(
        f'Dataset folder not found at {GDRIVE_DATASET}.\n'
        'Go to drive.google.com → New → Folder upload → select your "dataset" folder.'
    )

# Copy to Colab local disk for faster I/O during training
LOCAL_DATASET = '/content/water_meter/dataset'
if os.path.exists(LOCAL_DATASET):
    shutil.rmtree(LOCAL_DATASET)

print('Copying dataset from Drive to Colab disk (faster training I/O)...')
shutil.copytree(GDRIVE_DATASET, LOCAL_DATASET)
print('Done!')

# Show what we have
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(LOCAL_DATASET, 'images', split)
    lbl_dir = os.path.join(LOCAL_DATASET, 'labels', split)
    n_img = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    n_lbl = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 0
    print(f'  {split:5s}: {n_img} images, {n_lbl} labels')

## Step 3 — Fix data.yaml paths for Colab

In [ ]:
import yaml

DATA_YAML = '/content/water_meter/dataset/data.yaml'

with open(DATA_YAML) as f:
    data = yaml.safe_load(f)

# Override the Windows absolute path with the Colab path
data['path'] = '/content/water_meter/dataset'
data['train'] = 'images/train'
data['val']   = 'images/val'
data['test']  = 'images/test'

with open(DATA_YAML, 'w') as f:
    yaml.dump(data, f, default_flow_style=False, allow_unicode=True)

print('data.yaml updated:')
with open(DATA_YAML) as f:
    print(f.read())

## Step 4 — Train YOLOv8s with optimised settings

Key improvements over the baseline (which peaked at ~90% mAP@0.5):
- **yolov8s** (11.2M params) vs yolov8n (3.2M) — higher accuracy ceiling
- **No flips** — digits 6/9 and 2/5 are orientation-sensitive; flipping destroys class identity
- **imgsz=832** — resolves individual digits more sharply (GPU has the VRAM)
- **Cosine LR decay** — smoother convergence in final epochs
- **copy_paste=0.3** — synthesises more small-digit examples
- **patience=50** — keeps training until the model genuinely stalls

Results save to your Google Drive automatically — safe against Colab disconnects.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')  # ~22 MB download from Ultralytics

results = model.train(
    data=DATA_YAML,
    epochs=200,
    imgsz=832,
    batch=16,          # reduce to 8 if you get CUDA out-of-memory
    project='/content/drive/MyDrive/water_meter_runs',
    name='yolov8s_optimised',

    # Learning rate (cosine schedule)
    cos_lr=True,
    warmup_epochs=5,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,

    # Regularisation
    weight_decay=0.0005,
    momentum=0.937,

    # Early stopping
    patience=50,

    # Augmentation — NO flips (digit orientation matters!)
    fliplr=0.0,
    flipud=0.0,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.0005,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    mosaic=1.0,
    close_mosaic=15,
    copy_paste=0.3,
    mixup=0.0,

    workers=4,
    verbose=True,
)

map50 = results.results_dict.get('metrics/mAP50(B)', 0)
print(f'\nBest mAP@0.5: {map50:.4f}  (target ≥ 0.95)')

## Step 5 — Evaluate on the test set

In [ ]:
best_pt = '/content/drive/MyDrive/water_meter_runs/yolov8s_optimised/weights/best.pt'
best_model = YOLO(best_pt)

metrics = best_model.val(
    data=DATA_YAML,
    split='test',
    imgsz=832,
    conf=0.25,
    iou=0.5,
)

print('\n===== TEST SET RESULTS =====')
print(f'Precision:    {metrics.box.mp:.4f}')
print(f'Recall:       {metrics.box.mr:.4f}')
print(f'mAP@0.50:     {metrics.box.map50:.4f}  (target >= 0.95)')
print(f'mAP@0.5:0.95: {metrics.box.map:.4f}')

if metrics.box.map50 >= 0.95:
    print('\nTARGET REACHED — mAP@0.50 >= 95%')
else:
    gap = 0.95 - metrics.box.map50
    print(f'\nStill {gap:.3f} short of 95%. Run Step 4 again with --mode resume to continue training.')

## Step 6 — Download best.pt to your PC

In [ ]:
# Option A: download directly from this notebook
from google.colab import files
files.download(best_pt)

# Option B: it is already saved at:
# Google Drive → water_meter_runs/yolov8s_optimised/weights/best.pt

## After downloading

On your Windows PC, place `best.pt` here:
```
runs/detect/training_runs/yolov8s_optimised/weights/best.pt
```
Then run:
```bash
python 04_predict.py
```
`04_predict.py` auto-detects the newest `best.pt` and will use this model automatically.